# Feasibility of deeper Kronoberg transit analysis

A bounded pilot to choose the next investigation, rather than explain causes of delay.

September 2026 only, pinned dataset revision `3cb190163d9c400fd915bd62a919f8f9bfc90dd7`.
Run `uv run python scripts/prepare_case_study.py` first to retrieve this snapshot.
This notebook reads local Parquet only; it does not read credentials, fetch live data or write to Google Sheets.

The next investigation should compare weekday routes, directions and hours, then follow delay between timing stops. Reporting still requires at least 20 distinct observed service-date/trip pairs and 90% coverage.

## Load the frozen inputs

In [1]:
from pathlib import Path

import duckdb

root = Path.cwd()
if not (root / "pyproject.toml").exists() and (root.parent / "pyproject.toml").exists():
    root = root.parent
snapshot = root / "data/case-study/snapshot/data"
assert snapshot.exists(), "Run scripts/prepare_case_study.py from the repository root first."
connection = duckdb.connect()
for table in ["stop_events", "trips", "routes", "stops"]:
    pattern = (snapshot / table / "service_date=*" / "*.parquet").as_posix()
    assert len(list((snapshot / table).glob("service_date=*/*.parquet"))) == 30
    connection.execute(
        f"CREATE VIEW {table} AS SELECT * FROM read_parquet('{pattern}', hive_partitioning=true)"
    )
    dates = connection.execute(
        f"SELECT count(DISTINCT service_date),min(service_date),max(service_date) FROM {table}"
    ).fetchone()
    assert dates[0] == 30 and str(dates[1]) == "2026-09-01" and str(dates[2]) == "2026-09-30"
print("Four warehouse tables loaded; all 30 September dates are present.")

Four warehouse tables loaded; all 30 September dates are present.


## Population and joins

Departure hours use the scheduled departure time in Europe/Stockholm. Each stop event joins to its trip and route on service date as well as ID. Eligibility is restricted to in-scope, non-final timing-stop events with observed or unobserved status. Cancellation, skipped and DST-ambiguous departures therefore stay outside eligibility.

On time is −60 to +180 seconds inclusive. Rates are calculated from counts, never averaged across rows. Directions are GTFS codes 0 and 1, not presumed to mean inbound/outbound. Route-pattern variants must be checked in a subsequent operational study.

In [2]:
analysis_sql = """
CREATE TEMP VIEW base AS SELECT s.*, t.direction_id, r.route_short_name,
 extract(hour FROM timezone('Europe/Stockholm',timezone('UTC',s.scheduled_departure_utc))) AS hour_local,
 extract(isodow FROM s.service_date)<=5 AS weekday
 FROM stop_events s JOIN trips t USING(service_date,trip_id)
 JOIN routes r ON s.service_date=r.service_date AND s.route_id=r.route_id
 WHERE s.in_scope AND s.is_timing_stop AND s.stop_position<>'final'
 AND s.status IN ('observed','unobserved');
CREATE TEMP VIEW cells AS SELECT route_id,route_short_name,direction_id,hour_local,
 count(*) eligible, count(*) FILTER(WHERE status='observed') observed,
 count(DISTINCT (service_date,trip_id)) FILTER(WHERE status='observed') trips,
 count(*) FILTER(WHERE status='observed' AND delay_s BETWEEN -60 AND 180) on_time,
 count(*) FILTER(WHERE status='observed' AND delay_s>180) late,
 count(DISTINCT service_date) FILTER(WHERE status='observed') observed_dates,
 observed::DOUBLE/eligible coverage, on_time::DOUBLE/nullif(observed,0) punctuality
 FROM base WHERE weekday GROUP BY 1,2,3,4;
"""
connection.execute(analysis_sql)
print("Analysis views prepared.")

Analysis views prepared.


In [3]:
def show_query(sql):
    cursor = connection.execute(sql)
    names = [column[0] for column in cursor.description]
    rows = [dict(zip(names, row)) for row in cursor.fetchall()]
    displayed = names
    if "route_id" in names:
        displayed = [
            "route_short_name",
            "direction_id",
            "observed",
            "trips",
            "coverage_pct",
            "on_time_pct",
            "late",
        ]
    print(" | ".join(displayed))
    for row in rows:
        print(" | ".join(str(row[name]) for name in displayed))
    return rows

## Reconcile the primary population

In [4]:
headline = show_query(
    """SELECT count(*) eligible,count(*) FILTER(WHERE status='observed') observed,count(*) FILTER(WHERE delay_s BETWEEN -60 AND 180) on_time FROM base"""
)

eligible | observed | on_time
129069 | 127059 | 99396


## Can route × direction × hour groups meet the floor?

In [5]:
feasible_cells = show_query(
    """SELECT count(*) cell_count,count(*) FILTER(WHERE trips>=20 AND coverage>=0.9) reportable,count(*) FILTER(WHERE trips>=20 AND coverage>=0.9 AND hour_local BETWEEN 15 AND 17) afternoon_reportable FROM cells"""
)

cell_count | reportable | afternoon_reportable
1509 | 1265 | 301


## Illustrative candidates at 16:00–16:59 on weekdays

In [6]:
busy_16 = show_query(
    """SELECT route_short_name,route_id,direction_id,observed,trips,observed_dates,round(100*coverage,2) coverage_pct,round(100*punctuality,2) on_time_pct,late FROM cells WHERE hour_local=16 AND trips>=20 AND coverage>=0.9 ORDER BY late DESC,route_id,direction_id LIMIT 8"""
)

route_short_name | direction_id | observed | trips | coverage_pct | on_time_pct | late
3 | 1 | 489 | 207 | 98.99 | 54.6 | 221
1 | 0 | 514 | 173 | 98.28 | 58.56 | 209
1 | 1 | 384 | 173 | 98.97 | 48.44 | 198
7 | 1 | 323 | 151 | 99.38 | 55.73 | 143
3 | 0 | 340 | 192 | 99.42 | 60.88 | 133
5 | 0 | 193 | 86 | 98.97 | 44.04 | 107
2 | 1 | 156 | 78 | 97.5 | 33.33 | 103
840 | 1 | 131 | 44 | 99.24 | 24.43 | 99


## Size of the weekday afternoon population

In [7]:
afternoon = show_query(
    """SELECT count(*) eligible,count(*) FILTER(WHERE status='observed') observed,count(*) FILTER(WHERE status='observed' AND delay_s>180) late FROM base WHERE weekday AND hour_local BETWEEN 15 AND 17"""
)

eligible | observed | late
29687 | 29051 | 8294


## Can we follow a trip between timing stops?

In [8]:
pair_feasibility = show_query(
    """SELECT count(*) observed_trips,count(*) FILTER(WHERE n>=2) multi_timing_trips FROM (SELECT service_date,trip_id,count(*) n FROM base WHERE status='observed' GROUP BY 1,2)"""
)

observed_trips | multi_timing_trips
51279 | 39958


## Are direction codes available?

In [9]:
direction_check = show_query(
    """SELECT count(*) in_scope_trips,count(*) FILTER(WHERE direction_id IS NULL) missing_direction FROM trips WHERE in_scope"""
)

in_scope_trips | missing_direction
51714 | 0


## Check that joins preserve rows

In [10]:
joined = connection.execute("SELECT count(*) FROM base").fetchone()[0]
raw = connection.execute(
    "SELECT count(*) FROM stop_events WHERE in_scope AND is_timing_stop AND stop_position<>'final' AND status IN ('observed','unobserved')"
).fetchone()[0]
assert joined == raw == 129069
assert headline[0] == {"eligible": 129069, "observed": 127059, "on_time": 99396}
assert connection.execute("SELECT sum(observed) FROM cells").fetchone()[0] == 111587
print("Join row count, September headline and weekday counts reconcile.")

Join row count, September headline and weekday counts reconcile.


## Recommendation and limits

There are 1,265 reportable weekday route/direction/hour groups, including 301 across 15:00–17:59. Route 1's two direction groups at 16:00–16:59 show 58.56% and 48.44% punctuality, with coverage above 98% in both. Both groups have observations on all 22 September weekdays. These are investigation candidates, not evidence of congestion or operator fault.

For journey progression, 39,958 of 51,279 observed trips have at least two observed timing stops. This establishes feasibility, not representative pair coverage. A subsequent study must check adjacent scheduled timing-stop pairs, missing observations, patterns, and the known first-stop timestamp caveat. Final-stop departure punctuality remains outside the population.

Other supported investigations include recurring service patterns, severe-delay tails, station comparisons within the same route/hour, and missing realtime by route and scheduled time. Passenger impact and causal explanations need evidence absent from the current warehouse. No new dashboard or metric definition is introduced by this pilot.